In [9]:
# step 1: import semua yang dibutuhkan
import kagglehub
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np

load_dotenv()

True

In [8]:
# step 2: download competition
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')
print("Path to competition files:", path)

# cek dulu isinya
print("\nIsi folder:", os.listdir(path))

Path to competition files: /home/irzam/.cache/kagglehub/competitions/house-prices-advanced-regression-techniques

Isi folder: ['data_description.txt', 'sample_submission.csv', 'test.csv', 'train.csv']


In [23]:
# step 3: mulai eksplorasi data

# load data
df = pd.read_csv(os.path.join(path, 'train.csv'))

display(df.shape)                        # rows vs cols, notice anything about the ratio?
display(df.head())
display(df.dtypes.value_counts())        # how many numeric vs text columns

(1460, 81)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


str        43
int64      35
float64     3
Name: count, dtype: int64

- **Q (question)**: kolom apa yang dimiliki
- **S (saw)**: str=43, int64=35, float64=3
- **M (means)**:
    - cukup aneh, seharusnya banyak float karena banyak kolom yang berbau luas?
    - bisa jadi yang float itu banyak yang kosong?
- **D (do)**:
    - cek mana fitur kosong yang float

In [25]:
df.select_dtypes("float").columns   # names of the float columns

Index(['LotFrontage', 'MasVnrArea', 'GarageYrBlt'], dtype='str')

In [29]:
display(df.isna().mean().sort_values(ascending=False).head(20))  # worst missing columns
display(df.duplicated(subset=df.columns.drop("Id")).sum())       # duplicates ignoring Id

top_share = df.apply(lambda s: s.value_counts(normalize=True, dropna=False).iloc[0])  # share of most common value
display(top_share.sort_values(ascending=False).head(20))                             # near-constant columns

display(df.select_dtypes("number").columns)                      # numeric columns to sanity check

PoolQC          0.995205
MiscFeature     0.963014
Alley           0.937671
Fence           0.807534
MasVnrType      0.597260
FireplaceQu     0.472603
LotFrontage     0.177397
GarageQual      0.055479
GarageFinish    0.055479
GarageType      0.055479
GarageYrBlt     0.055479
GarageCond      0.055479
BsmtFinType2    0.026027
BsmtExposure    0.026027
BsmtCond        0.025342
BsmtQual        0.025342
BsmtFinType1    0.025342
MasVnrArea      0.005479
Electrical      0.000685
Condition2      0.000000
dtype: float64

np.int64(0)

Utilities       0.999315
Street          0.995890
PoolArea        0.995205
PoolQC          0.995205
Condition2      0.989726
3SsnPorch       0.983562
RoofMatl        0.982192
LowQualFinSF    0.982192
Heating         0.978082
MiscVal         0.964384
MiscFeature     0.963014
KitchenAbvGr    0.953425
LandSlope       0.946575
BsmtHalfBath    0.943836
Alley           0.937671
CentralAir      0.934932
Functional      0.931507
ScreenPorch     0.920548
PavedDrive      0.917808
Electrical      0.913699
dtype: float64

Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
       'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
       'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
       'MiscVal', 'MoSold', 'YrSold', 'SalePrice'],
      dtype='str')

- **Q**: berapa jumlah kolom yang:
    - kebanyakan kosong
    - duplikat
    - value hampir semua sama
    - masuk numerik
- **S**:
    - ada sekitar 7 kolom yang memiliki value kosong di atas 15% dan 3 teratas kosong >99%; 11 kolom setelahnya berkisar antara 0.5-6% dan sisanya di bawah 0.1%
    - tidak ada data duplikat
    - top 20 memiliki >90% kesamaan value, dan top 50 >50%
- **M**:
    - hanya sebagian kecil fitur yang memiliki banyak value kosong
    - banyak data sama di banyak fitur namun tidak ada yang duplikat
- **D**:
    - no idea

In [ ]:
# Q: kolom mana yang banyak kosong, dan kenapa?
df.isna().mean().sort_values(ascending=False).head(20)